In [ ]:
# Базовые библиотеки
import numpy as np
import pandas as pd

# Визуализация
import matplotlib.pyplot as plt

# Разбиение и CV
from sklearn.model_selection import train_test_split, GridSearchCV

# Baseline
from sklearn.dummy import DummyClassifier

# Линейная модель
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Деревья и ансамбли
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

# Метрики
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score
)

# Интерпретация
from sklearn.inspection import permutation_importance

# Сохранение артефактов
import json
import joblib
from pathlib import Path

# Фиксация стиля графиков
plt.rcParams["figure.figsize"] = (8, 6)


In [ ]:
BASE_DIR = Path("homeworks/HW06")
ARTIFACTS_DIR = BASE_DIR / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from google.colab import files;
files.upload()

In [ ]:
df = pd.read_csv("S06-hw-dataset-04.csv")
print(f"Файл успешно загружен. Размер датасета: {df.shape}")

In [ ]:
# Первичный анализ данных (EDA)

df.info()
df.describe()
# Проверка пропусков
df.isna().sum()
# Распределение целевой переменной
target_dist = df["target"].value_counts(normalize=True)
target_dist

In [ ]:
# Формирование X и Y
X = df.drop(columns=["target", "id"], errors="ignore")
y = df["target"]

X.shape, y.shape

In [ ]:
# Train / Test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

In [ ]:
# Общая функция оценки модели

def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    print(f"\n{name}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    return metrics, y_pred, y_proba

In [ ]:
# Baseline 1: DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_metrics, _, _ = evaluate_model(
    dummy, X_test, y_test, "DummyClassifier"
)

In [ ]:
# Baseline 2: LogisticRegression (Pipeline)

logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

logreg_pipe.fit(X_train, y_train)

logreg_metrics, _, _ = evaluate_model(
    logreg_pipe, X_test, y_test, "LogisticRegression"
)

In [ ]:
# DecisionTreeClassifier + GridSearchCV

dt = DecisionTreeClassifier(random_state=42)

dt_param_grid = {
    "max_depth": [5, 10],
    "min_samples_leaf": [5, 10]
}

dt_search = GridSearchCV(
    dt,
    dt_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

dt_search.fit(X_train, y_train)

best_dt = dt_search.best_estimator_


In [ ]:
dt_metrics, _, _ = evaluate_model(
    best_dt, X_test, y_test, "DecisionTreeClassifier"
)

In [ ]:
# RandomForestClassifier + GridSearchCV

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

rf_param_grid = {
    "max_depth": [None, 10],
    "min_samples_leaf": [5],
    "max_features": ["sqrt"]
}

rf_search = GridSearchCV(
    rf,
    rf_param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=1
)

rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_

In [ ]:
rf_metrics, rf_pred, rf_proba = evaluate_model(
    best_rf, X_test, y_test, "RandomForestClassifier"
)

In [ ]:
# GradientBoostingClassifier + GridSearchCV

from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)

hgb_param_grid = {
    "learning_rate": [0.1],
    "max_depth": [3, 5],
    "max_iter": [100]
}

hgb_search = GridSearchCV(
    hgb,
    hgb_param_grid,
    cv=3,
    scoring="roc_auc"
)

hgb_search.fit(X_train, y_train)

best_gb = hgb_search.best_estimator_

In [ ]:
gb_metrics, gb_pred, gb_proba = evaluate_model(
    best_gb, X_test, y_test, "GradientBoostingClassifier"
)

In [ ]:
# Сравнение метрик и выбор лучшей модели

metrics_test = {
    "DummyClassifier": dummy_metrics,
    "LogisticRegression": logreg_metrics,
    "DecisionTree": dt_metrics,
    "RandomForest": rf_metrics,
    "GradientBoosting": gb_metrics
}

metrics_test

In [ ]:
best_model_name = max(metrics_test, key=lambda k: metrics_test[k]["roc_auc"])
best_model_name

In [ ]:
best_model = {
    "RandomForest": best_rf,
    "GradientBoosting": best_gb,
    "DecisionTree": best_dt
}[best_model_name]

In [ ]:
# ROC-кривая (АРТЕФАКТ)

fpr, tpr, _ = roc_curve(y_test, rf_proba)

plt.plot(fpr, tpr, label="RandomForest")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig(FIGURES_DIR / "roc_curve.png")
plt.show()

In [ ]:
# Confusion Matrix (АРТЕФАКТ)

cm = confusion_matrix(y_test, rf_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("Confusion Matrix")
plt.savefig(FIGURES_DIR / "confusion_matrix.png")
plt.show()

In [ ]:
# PR-кривая (для дисбаланса)

precision, recall, _ = precision_recall_curve(y_test, rf_proba)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.savefig(FIGURES_DIR / "pr_curve.png")
plt.show()

In [ ]:
# Permutation Importance

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importances = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

top_features = importances.head(15)
top_features

In [ ]:
top_features.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Permutation Importance (Top-15)")
plt.savefig(FIGURES_DIR / "permutation_importance.png")
plt.show()

In [ ]:
# Сохранение артефактов (ОБЯЗАТЕЛЬНО)

# metrics_test.json
with open(ARTIFACTS_DIR / "metrics_test.json", "w") as f:
    json.dump(metrics_test, f, indent=4)

In [ ]:
# search_summaries.json
search_summaries = {
    "DecisionTree": dt_search.best_params_,
    "RandomForest": rf_search.best_params_,
    "GradientBoosting": gb_search.best_params_
}

with open(ARTIFACTS_DIR / "search_summaries.json", "w") as f:
    json.dump(search_summaries, f, indent=4)

In [ ]:
# best_model.joblib
joblib.dump(best_model, ARTIFACTS_DIR / "best_model.joblib")

In [ ]:
# best_model_meta.json
best_model_meta = {
    "best_model": best_model_name,
    "metrics_test": metrics_test[best_model_name]
}

with open(ARTIFACTS_DIR / "best_model_meta.json", "w") as f:
    json.dump(best_model_meta, f, indent=4)